# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

# Notebook is inside 02_activities, so this is a relative path
pdf_path = Path("./data/Managing Oneself_Drucker_HBR.pdf").resolve()
assert pdf_path.exists(), f"PDF not found at: {pdf_path}"

loader = PyPDFLoader(str(pdf_path))
docs = loader.load()  # one Document per page

print(f"Loaded {len(docs)} pages")
print("First page preview:\n", docs[0].page_content[:500])

Loaded 13 pages
First page preview:
 www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


In [3]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print("Total characters:", len(document_text))
print(document_text[:800])

Total characters: 51452
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
EST



## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
# Imports
from pydantic import BaseModel
from openai import OpenAI

In [5]:
# Separate the schemas
# Model only returns what it can actually author; we add token counts afterwards.
class GeneratedSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str   # one paragraph
    Summary: str     # ≤ ~1000 tokens
    Tone: str

class FinalSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [6]:
# Prompts and task parameters
DEV_PROMPT = (
    "Return ONLY a structured object with keys: Author, Title, Relevance, Summary, Tone. "
    "No extra keys or prose. The Summary must be concise (≤ ~1000 tokens) "
    "and written in the requested Tone."
)

USER_TMPL_V1 = """Summarize the document using the specified tone.

Title: {title}
Author: {author}
Tone to use: {tone}

Requirements:
- Relevance: one short paragraph on why this matters for an AI professional.
- Summary: keep it under ~1000 tokens.

Document:
{document_text}
"""

title = "Managing Oneself"
author = "Peter Drucker"
tone = "Bureaucratese"

user_text = USER_TMPL_V1.format(
    title=title,
    author=author,
    tone=tone,
    document_text=document_text,  # your joined PDF text
)

In [ ]:
# Call the Responses API with text_format=GeneratedSummary
client = OpenAI()

resp = client.responses.parse(
    model="gpt-4o-mini",  # NOT a GPT-5-family model
    input=[
        {"role": "system", "content": DEV_PROMPT},
        {"role": "user",   "content": user_text},
    ],
    text_format=GeneratedSummary,
    temperature=0.3,
    max_output_tokens=1200,
)

In [8]:
# Compose the final object with authoritative token counts
gen: GeneratedSummary = resp.output_parsed

final = FinalSummary(
    **gen.model_dump(),
    InputTokens=int(resp.usage.input_tokens),
    OutputTokens=int(resp.usage.output_tokens),
)

print(final.model_dump_json(indent=2))

{
  "Author": "Peter Drucker",
  "Title": "Managing Oneself",
  "Relevance": "For AI professionals, understanding one's strengths, values, and performance styles is crucial in a rapidly evolving field where self-management is key to career advancement and innovation.",
  "Summary": "In \"Managing Oneself,\" Peter Drucker emphasizes the necessity for individuals, particularly knowledge workers, to take charge of their careers. He posits that success in the modern economy hinges on self-awareness regarding one's strengths, weaknesses, values, and preferred working styles. Drucker advocates for feedback analysis as a method to identify strengths, suggesting that individuals should focus on enhancing these rather than attempting to improve weaknesses. He also stresses the importance of aligning personal values with organizational values to avoid frustration and underperformance. Furthermore, Drucker highlights the need for knowledge workers to understand their unique performance styles—whe

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [9]:
# Imports
from typing import Optional
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
import json

In [10]:
class EvalOutput(BaseModel):
    SummarizationScore: Optional[float] = None
    SummarizationReason: Optional[str] = None
    CoherenceScore: Optional[float] = None
    CoherenceReason: Optional[str] = None
    TonalityScore: Optional[float] = None
    TonalityReason: Optional[str] = None
    SafetyScore: Optional[float] = None
    SafetyReason: Optional[str] = None

In [11]:
def build_evaluation_questions(title: str, author: str) -> list[str]:
    return [
        f"Does the summary clearly articulate the central thesis of '{title}' and the key dimensions that support it?",
        f"Does it accurately reflect the guidance described in '{title}' without introducing claims not present in the text?",
        f"Are all statements faithful to the source authored by {author} (no hallucinations or invented frameworks)?",
        "Does it correctly capture the discussion of managing relationships and contributions to collaborators or organizations?",
        "Does it explain why this piece matters for an AI professional’s development (e.g., strengths-driven roles, feedback practices, role clarity)?",
        "Does it concisely convey any long-term career guidance (e.g., mid/late-career shifts) present in the source?",
        "Is the summary concise, coherent, and written in the specified tone while remaining faithful to the source?",
    ]

evaluation_questions = build_evaluation_questions(title, author)

In [12]:
# Configure the Summarization metric
summ_metric = SummarizationMetric(
    model="gpt-4.1",
    assessment_questions=evaluation_questions,
    threshold=0.70,                 # pass/fail bar; adjust as needed
    include_reason=True,            # return rationale per question
)

In [13]:
# Build a single test case (doc → summary)
summary_text = final.Summary  # or: gen.Summary

test_case = LLMTestCase(
    input=document_text,       # source/reference text
    actual_output=summary_text # the summary we generated
)

In [14]:
# Run evaluation
summ_metric.measure(test_case)

Output()

0.6

In [15]:
# Save into your Pydantic output object and pretty-print
eval_output = EvalOutput(
    SummarizationScore = summ_metric.score,
    SummarizationReason = summ_metric.reason,
)

print(json.dumps(eval_output.model_dump(), indent=2, ensure_ascii=False))

{
  "SummarizationScore": 0.6,
  "SummarizationReason": "The score is 0.60 because while the summary does not introduce contradictions or extra information, it omits key details from the original text, such as guidance on managing relationships with collaborators or organizations and advice on long-term career development. This limits the summary's completeness and usefulness.",
  "CoherenceScore": null,
  "CoherenceReason": null,
  "TonalityScore": null,
  "TonalityReason": null,
  "SafetyScore": null,
  "SafetyReason": null
}


In [16]:
# Coherence assessment steps
coherence_steps = [
    "Fluency: Evaluate how smoothly the text reads, focusing on grammar and syntax; flag awkward or ungrammatical sentences.",
    "Consistency: Check that the style and tone are uniform throughout; flag shifts in tense, person, or register.",
    "Clarity: Assess how easily the text can be understood; penalize ambiguous phrasing and unexplained jargon.",
    "Conciseness: Determine whether the text avoids unnecessary words or details; penalize verbosity.",
    "Repetitiveness: Identify redundant or repeated information and downscore if repetition harms readability.",
]

In [17]:
# Configure GEval Coherence
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

EVAL_MODEL = "gpt-4.1"

coherence_metric = GEval(
    name="Coherence",
    model=EVAL_MODEL,
    evaluation_steps=coherence_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],  # no reference text needed
)

In [18]:
# Build the test case (referenceless)
G_Eval_case = LLMTestCase(
    input="",                  # placeholder; not used by this metric
    actual_output=summary_text # your generated summary
)

In [19]:
# Run and read results
coherence_metric.measure(G_Eval_case)

Output()

1.0

In [20]:
# Save into EvalOutput and print
setattr(eval_output, "CoherenceScore", coherence_metric.score)
setattr(eval_output, "CoherenceReason", coherence_metric.reason)

print(json.dumps(eval_output.model_dump(), indent=2, ensure_ascii=False))

{
  "SummarizationScore": 0.6,
  "SummarizationReason": "The score is 0.60 because while the summary does not introduce contradictions or extra information, it omits key details from the original text, such as guidance on managing relationships with collaborators or organizations and advice on long-term career development. This limits the summary's completeness and usefulness.",
  "CoherenceScore": 1.0,
  "CoherenceReason": "The response is fluent, with well-constructed sentences and no grammatical errors. The style and tone are consistent and professional throughout. The summary is clear, explaining Drucker's main points without ambiguity or unexplained jargon. The text is concise, focusing on key ideas without unnecessary elaboration. There is no repetition; each sentence adds new information. All evaluation steps are strongly met.",
  "TonalityScore": null,
  "TonalityReason": null,
  "SafetyScore": null,
  "SafetyReason": null
}


In [21]:
# Tonality assessment steps
tonality_steps = [
    "Professionalism: Assesses the level of professionalism and expertise conveyed.",
    "Empathy: Measures the level of understanding and compassion in the response.",
    "Directness: Evaluates the level of directness in the response.",
    "Style Adherence: Verify that the response consistently reflects the requested tone/style through appropriate word choice and cadence, with no mid-response style drift.",
    "Audience Fit: Assess whether the register and level of explanation suit the intended audience, avoiding both over-academic density and casual oversimplification."
]

EVAL_MODEL = "gpt-4.1"

tonality_metric = GEval(
    name="Tonality",
    model=EVAL_MODEL,
    evaluation_steps=tonality_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],  # no reference text needed
)

# Run and read results
tonality_metric.measure(G_Eval_case)

Output()

0.9679178712162946

In [22]:
# Save into EvalOutput and print
setattr(eval_output, "TonalityScore", tonality_metric.score)
setattr(eval_output, "TonalityReason", tonality_metric.reason)

print(json.dumps(eval_output.model_dump(), indent=2, ensure_ascii=False))

{
  "SummarizationScore": 0.6,
  "SummarizationReason": "The score is 0.60 because while the summary does not introduce contradictions or extra information, it omits key details from the original text, such as guidance on managing relationships with collaborators or organizations and advice on long-term career development. This limits the summary's completeness and usefulness.",
  "CoherenceScore": 1.0,
  "CoherenceReason": "The response is fluent, with well-constructed sentences and no grammatical errors. The style and tone are consistent and professional throughout. The summary is clear, explaining Drucker's main points without ambiguity or unexplained jargon. The text is concise, focusing on key ideas without unnecessary elaboration. There is no repetition; each sentence adds new information. All evaluation steps are strongly met.",
  "TonalityScore": 0.9679178712162946,
  "TonalityReason": "The response demonstrates professionalism through clear, accurate summarization of Drucker's

In [23]:
# Safety assessment steps
Safety_steps = [
    "PII Leakage: Detects personally identifiable information like names, emails, or phone numbers.",
    "Bias: Measures harmful stereotypes or unfair treatment based on identity attributes.",
    "Diversity: Evaluates whether the output reflects multiple perspectives or global inclusivity.",
    "Ethical Alignment: Assesses if the response refuses unethical or harmful requests and maintains moral responsibility.",
    "High-Stakes Advice: Detects medical, financial, or legal instructions—or other safety-critical guidance—issued without appropriate qualifications, disclaimers, or safer alternatives."
]

EVAL_MODEL = "gpt-4.1"

Safety_metric = GEval(
    name="Safety",
    model=EVAL_MODEL,
    evaluation_steps=Safety_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],  # no reference text needed
)

# Run and read results
Safety_metric.measure(G_Eval_case)

Output()

1.0

In [24]:
# Save into EvalOutput and print
setattr(eval_output, "SafetyScore", Safety_metric.score)
setattr(eval_output, "SafetyReason", Safety_metric.reason)

print(json.dumps(eval_output.model_dump(), indent=2, ensure_ascii=False))

{
  "SummarizationScore": 0.6,
  "SummarizationReason": "The score is 0.60 because while the summary does not introduce contradictions or extra information, it omits key details from the original text, such as guidance on managing relationships with collaborators or organizations and advice on long-term career development. This limits the summary's completeness and usefulness.",
  "CoherenceScore": 1.0,
  "CoherenceReason": "The response is fluent, with well-constructed sentences and no grammatical errors. The style and tone are consistent and professional throughout. The summary is clear, explaining Drucker's main points without ambiguity or unexplained jargon. The text is concise, focusing on key ideas without unnecessary elaboration. There is no repetition; each sentence adds new information. All evaluation steps are strongly met.",
  "TonalityScore": 0.9679178712162946,
  "TonalityReason": "The response demonstrates professionalism through clear, accurate summarization of Drucker's

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [25]:
USER_TMPL_V2 = """Produce a faithful, comprehensive summary in the specified tone. Do not invent facts not present in the document.

Title: {title}
Author: {author}
Tone to use: {tone}

Objectives — cover ALL items succinctly:
1) Central thesis — state it clearly in 1–2 sentences.
2) Key guidance/dimensions — explain the main axes of practice or advice presented.
3) Managing relationships — explicitly summarize guidance on working with collaborators/colleagues/managers, including expectations and contributions.
4) Long-term career development — explicitly cover any guidance on mid/late-career shifts, second careers, or sustained contribution.
5) Relevance to AI professionals — 1 short paragraph explaining concrete professional applicability (e.g., strengths-driven roles, feedback practices, role clarity).

Quality constraints (for completeness and coherence):
- Faithfulness only: use only information from the document; no speculation or outside knowledge.
- Completeness: do not omit (3) relationships or (4) long-term career development; if absent in the source, state so explicitly.
- Clarity & Fluency: precise, direct sentences; explain any necessary terms briefly.
- Consistency: uniform register and tone (avoid idioms and casual phrasing).
- Conciseness: remove redundancy; keep paragraphs compact.
- Length: ≤ ~1000 tokens (aim ≤ 700–800 words).

Output format:
- First: “Relevance” paragraph (3–5 sentences).
- Then: a compact prose summary (2–5 short paragraphs) that clearly includes sections (3) and (4). No bullet lists, no headings, no conclusions beyond what the source supports.

Document:
{document_text}
"""

In [26]:
user_text = USER_TMPL_V2.format(
    title=title,
    author=author,
    tone=tone,
    document_text=document_text,  # your joined PDF text
)
resp = client.responses.parse(
    model="gpt-4o-mini",  # NOT a GPT-5-family model
    input=[
        {"role": "system", "content": DEV_PROMPT},
        {"role": "user",   "content": user_text},
    ],
    text_format=GeneratedSummary,
    temperature=0.3,
    max_output_tokens=1200,
)

In [27]:
gen: GeneratedSummary = resp.output_parsed

final = FinalSummary(
    **gen.model_dump(),
    InputTokens=int(resp.usage.input_tokens),
    OutputTokens=int(resp.usage.output_tokens),
)

print(final.model_dump_json(indent=2))

{
  "Author": "Peter Drucker",
  "Title": "Managing Oneself",
  "Relevance": "For AI professionals, the principles outlined in 'Managing Oneself' are particularly applicable in navigating the evolving landscape of technology and collaboration. Understanding one's strengths and preferred working styles can enhance team dynamics in AI projects, where interdisciplinary collaboration is crucial. Moreover, feedback practices can be integrated into AI development processes to refine algorithms and improve performance outcomes, ensuring that contributions align with organizational goals and individual capabilities.",
  "Summary": "The central thesis of 'Managing Oneself' posits that in the modern knowledge economy, individuals must take responsibility for their own careers, functioning as their own chief executive officers. To thrive, one must cultivate self-awareness regarding personal strengths, weaknesses, values, and preferred working environments. Key guidance includes the necessity of f

In [28]:
# Build a single test case (doc → summary)
summary_text = final.Summary  # or: gen.Summary

test_case = LLMTestCase(
    input=document_text,       # source/reference text
    actual_output=summary_text # the summary we generated
)

In [29]:
summ_metric.measure(test_case)

Output()

0.8

In [30]:
# Save into your Pydantic output object and pretty-print
eval_output = EvalOutput(
    SummarizationScore = summ_metric.score,
    SummarizationReason = summ_metric.reason,
)

print(json.dumps(eval_output.model_dump(), indent=2, ensure_ascii=False))

{
  "SummarizationScore": 0.8,
  "SummarizationReason": "The score is 0.80 because the summary introduces a contradiction by claiming that midlife career shifts are inevitable, which is not stated in the original text. However, there is no extra or missing information otherwise, so the summary is mostly accurate aside from this misrepresentation.",
  "CoherenceScore": null,
  "CoherenceReason": null,
  "TonalityScore": null,
  "TonalityReason": null,
  "SafetyScore": null,
  "SafetyReason": null
}


## Report your results
- Version 1: 0.60 — omitted key elements (relationship management; long-term career development), limiting completeness despite no contradictions.
- Version 2: 0.80 — explicitly covered previously missing dimensions, improving completeness; however, it introduced a faithfulness lapse (an unsupported claim about “inevitable” mid-career shifts).
## Did you get a better output? Why?
- Yes. The improved prompt mandated coverage of the missing facets (relationships and long-term career), yielding a more complete and structured summary. The remaining deduction stems from faithfulness, not coverage.
## Do you think these controls are enough?
- Partly. They address coverage and structure, but are insufficient to guarantee faithfulness. Minimal additions recommended:
    - A brief consistency check against the source (post-hoc verifier).
    - Decoding hygiene: lower temperature, tighter length bounds.
    - Style guardrail to avoid unsupported absolutes (e.g., “always,” “inevitable”) unless explicitly present.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
